# Bab 7: How to Picture Neural Networks — In Your Head and on Paper

Notebook ini merangkum Bab 7 buku *Grokking Deep Learning* karya Andrew W. Trask. Bab ini tidak memperkenalkan arsitektur atau eksperimen baru. Isinya cara merangkum semua pelajaran sebelumnya menjadi gambaran mental yang ringkas, supaya kita bisa merancang arsitektur baru, men-debug eksperimen, dan memakai arsitektur untuk masalah lain tanpa tenggelam dalam detail.

Isi notebook:

1. Meninjau ulang konsep yang sudah dipelajari
2. Correlation summarization
3. Visualisasi yang disederhanakan (vektor dan matriks sebagai balok LEGO)
4. Melihat network melakukan prediksi
5. Visualisasi dengan huruf (aljabar)
6. Semua cara pandang dalam satu tempat
7. Pentingnya alat visualisasi

## 1. Saatnya Menyederhanakan

Tidak praktis memikirkan semuanya sekaligus. Kita butuh alat mental. Bab 6 ditutup dengan kode network sepanjang 35 baris yang sangat padat, dan kode itu menyimpan lebih dari 100 halaman konsep. Penulis menyarankan agar contoh-contoh ini terus dibangun ulang dari ingatan. Seiring contoh membesar, latihan ini bukan lagi soal mengingat huruf demi huruf kode, tetapi mengingat **konsep** lalu menyusun kodenya dari konsep itu.

Ringkasan perjalanan buku sampai sini:

1. Ide dasar machine learning.
2. Cara satu neuron linear belajar.
3. Kelompok neuron horizontal (layer) dan vertikal (tumpukan layer).
4. Belajar berarti menurunkan error sampai 0, dan kalkulus dipakai untuk mencari arah perubahan tiap weight.
5. Network mencari (dan kadang menciptakan) korelasi antara dataset input dan output.

Poin terakhir itu merangkum semua yang sebelumnya: neuron, gradien, dan tumpukan layer pada akhirnya mengarah ke satu gagasan, yaitu **neural network menemukan dan menciptakan korelasi**. Memegang ide korelasi ini lebih berguna daripada mengingat semua bagian kecilnya, kalau tidak ingin kewalahan oleh kompleksitas network.

## 2. Correlation Summarization

> **Correlation summarization**: neural network mencari korelasi langsung maupun tidak langsung antara layer input dan layer output, yang ditentukan oleh dataset input dan output.

Itu pandangan dari ketinggian 10.000 kaki. Kalau kita zoom sedikit ke satu weight matrix tertentu, ada dua level:

> **Local correlation summarization**: setiap kumpulan weight belajar mengorelasikan layer inputnya dengan apa yang menurut layer output seharusnya terjadi.

> **Global correlation summarization**: apa yang seharusnya dimiliki sebuah layer awal bisa ditentukan dari apa yang diinginkan layer berikutnya, dikalikan weight di antara keduanya. Dengan cara ini, layer belakang memberi tahu layer depan sinyal seperti apa yang mereka butuhkan agar akhirnya berkorelasi dengan output. Komunikasi silang ini disebut **backpropagation**.

Analogi dari buku: permainan telepon berantai. Neuron di layer terakhir bilang, "Saya perlu sedikit lebih tinggi," lalu meneruskannya ke layer sebelumnya, "Kirim sinyal yang lebih tinggi," dan seterusnya. Di ujung permainan, tiap layer tahu neuronnya mana yang perlu naik atau turun, lalu local correlation summarization mengambil alih untuk memperbarui weight.

Konsekuensi praktisnya: kita tidak perlu lagi memikirkan bagaimana tiap weight diperbarui. Layer belakang sudah tahu cara memberi tahu layer depan apakah perlu sinyal lebih tinggi atau lebih rendah. Yang kita pedulikan hanya apakah weight berperilaku semestinya, yaitu menangkap korelasi dengan cara yang bisa digeneralisasi.

## 3. Visualisasi yang Terlalu Rumit di Awal

Selama ini kita menggambar network sebagai kumpulan node yang dihubungkan banyak garis. Gambar itu berguna untuk memahami dasar: weight matrix adalah garis-garis antar node, dan layer adalah deretan node (vektor). Misalnya `weights_1_2` adalah matriks, `weights_0_1` adalah matriks, dan `layer_1` adalah vektor.

Tetapi gambar itu terlalu detail untuk dipakai ke depan. Kalau layer berisi 500 node, menggambar setiap garis mustahil dan tidak membantu. Jadi kita beralih ke pandangan umum: vektor dan matriks berukuran sembarang.

## 4. Visualisasi yang Disederhanakan

Neural network itu seperti balok LEGO, dan setiap balok adalah vektor atau matriks. Ide korelasi tidak bergantung pada susunan balok tertentu: backpropagation, gradient descent, alpha, dropout, mini-batching, semuanya tetap berlaku apa pun susunannya.

Konvensi gambar yang dipakai:

- Garis tipis/strip = **vektor**
- Kotak = **matriks** (bisa dilihat sebagai "vektor dari vektor")
- Lingkaran = bobot/angka individual

Dengan konvensi ini, network streetlight dari Bab 6 digambarkan sebagai berikut (ukuran di dalam kurung adalah baris × kolom):

| Komponen | Bentuk |
|---|---|
| `layer_0` | 1 × 3 |
| `weights_0_1` | 3 × 4 |
| `layer_1` | 1 × 4 |
| `weights_1_2` | 4 × 1 |
| `layer_2` | 1 × 1 |

Kita bisa memastikannya lewat NumPy.

In [1]:
import numpy as np

np.random.seed(1)

layer_0 = np.array([[1, 0, 1]])                          # 1 x 3
weights_0_1 = 2 * np.random.random((3, 4)) - 1          # 3 x 4
weights_1_2 = 2 * np.random.random((4, 1)) - 1          # 4 x 1

print("layer_0     :", layer_0.shape)
print("weights_0_1 :", weights_0_1.shape)
print("weights_1_2 :", weights_1_2.shape)

layer_0     : (1, 3)
weights_0_1 : (3, 4)
weights_1_2 : (4, 1)


## 5. Menyederhanakan Lebih Jauh

Dimensi sebuah matriks ditentukan oleh layer di sebelah kiri dan kanannya. `weights_0_1` pasti berukuran 3 × 4 karena `layer_0` punya 3 dimensi dan `layer_1` punya 4 dimensi. Agar ada satu weight untuk tiap pasangan node dari `layer_0` ke `layer_1`, matriksnya harus berukuran persis itu. Jadi gambar tanpa angka ukuran pun sudah memuat semua informasi untuk membangun network.

Pola ini membuat kita bisa berpikir dengan correlation summarization: network hanya akan menyesuaikan weight untuk mencari korelasi antara `layer_0` dan `layer_2`, memakai semua metode yang sudah dibahas. Namun susunan layer dan weight di antara input dan output sangat memengaruhi apakah network berhasil menemukan korelasi, dan seberapa cepat.

Susunan layer dan weight tertentu disebut **arsitektur**. Sebagian besar sisa buku membahas kelebihan dan kekurangan berbagai arsitektur.

> Arsitektur neural yang baik mengalirkan sinyal sehingga korelasi mudah ditemukan. Arsitektur yang hebat juga menyaring noise untuk membantu mencegah overfitting.

## 6. Melihat Network Ini Memprediksi

Ikuti satu datapoint dari dataset streetlight melewati network:

1. **Gambar 1**: satu datapoint dipilih dan `layer_0` diisi dengan nilainya, misalnya `[1, 0, 1]`.
2. **Gambar 2**: empat weighted sum dari `layer_0` dihitung oleh `weights_0_1`. Proses ini adalah perkalian vektor-matriks. Empat nilai itu masuk ke empat posisi `layer_1` dan dilewatkan ke fungsi `relu` (nilai negatif jadi 0). Misalnya nilai ketiga seharusnya negatif, tapi `relu` menjadikannya 0.
3. **Gambar 3**: langkah terakhir adalah weighted sum dari `layer_1` memakai perkalian vektor-matriks lagi, menghasilkan prediksi akhir.

> **Review perkalian vektor-matriks**: operasi ini melakukan beberapa weighted sum dari sebuah vektor. Matriks harus punya jumlah baris yang sama dengan jumlah nilai pada vektor, supaya tiap kolom matriks menghasilkan satu weighted sum yang unik. Matriks dengan empat kolom menghasilkan empat weighted sum.

In [2]:
def relu(x):
    return (x > 0) * x

# Langkah 1: layer_0 diisi satu datapoint
layer_0 = np.array([[1, 0, 1]])
print("layer_0 :", layer_0)

# Langkah 2: perkalian vektor-matriks (4 weighted sum), lalu relu
layer_1_raw = layer_0.dot(weights_0_1)
layer_1 = relu(layer_1_raw)
print("sebelum relu:", np.round(layer_1_raw, 3))
print("sesudah relu:", np.round(layer_1, 3))

# Langkah 3: perkalian vektor-matriks lagi untuk prediksi akhir
layer_2 = layer_1.dot(weights_1_2)
print("prediksi layer_2:", np.round(layer_2, 3))

layer_0 : [[1 0 1]]
sebelum relu: [[-0.372  0.518 -1.161 -0.025]]
sesudah relu: [[-0.     0.518 -0.    -0.   ]]
prediksi layer_2: [[0.392]]


Angka weight di atas acak (dengan seed tetap), jadi nilai prediksinya bukan prediksi yang "benar". Intinya ada pada bentuk operasinya: dua perkalian vektor-matriks dengan `relu` di antaranya.

## 7. Visualisasi dengan Huruf, Bukan Gambar

Semua gambar dan penjelasan tadi ternyata aljabar sederhana. Sama seperti kita menyederhanakan matriks dan vektor menjadi gambar, kita bisa menyederhanakannya menjadi huruf:

- **Matriks** = huruf kapital. Penulis memilih **W** (untuk *weights*). Angka kecil di bawahnya (W₀, W₁) membedakan beberapa matriks weight.
- **Vektor** = huruf kecil. Penulis memilih **l** (untuk *layer*), diberi nomor: l₀, l₁, l₂.

Huruf apa pun boleh dipakai. Ini visualisasi kita sendiri, jadi pilih yang mudah diingat.

## 8. Menghubungkan Variabel

Untuk menunjukkan dua huruf dikalikan, letakkan berdampingan:

| Aljabar | Terjemahan |
|---|---|
| `l0 W0` | Ambil vektor layer 0 dan lakukan perkalian vektor-matriks dengan weight matrix 0. |
| `l1 W1` | Ambil vektor layer 1 dan lakukan perkalian vektor-matriks dengan weight matrix 1. |

Fungsi sembarang seperti `relu` bisa ditulis dengan notasi yang hampir sama dengan kode Python:

| Aljabar | Terjemahan |
|---|---|
| `l1 = relu(l0 W0)` | Buat layer 1 dengan perkalian vektor-matriks layer 0 dan W0, lalu terapkan `relu` pada hasilnya. |
| `l2 = l1 W1` | Buat layer 2 dengan perkalian vektor-matriks layer 1 dan W1. |

Karena layer 2 memakai layer 1 sebagai input, seluruh network bisa ditulis dalam satu ekspresi:

**`l2 = relu(l0 W0) W1`**

Semua logika forward propagation ada dalam rumus itu. Rumus tersebut mengasumsikan dimensi vektor dan matriks sudah cocok.

## 9. Semua Cara Pandang Berdampingan

Satu hal yang sama, dilihat dengan empat cara:

- **Gambar**: input, hidden, prediksi, dihubungkan `weights_0_1` dan `weights_1_2`.
- **Kode Python**: `layer_2 = relu(layer_0.dot(weights_0_1)).dot(weights_1_2)`
- **Aljabar**: `l2 = relu(l0 W0) W1`
- **Bentuk matriks**: (1 × 3) · (3 × 4) → (1 × 4) → (1 × 4) · (4 × 1) → (1 × 1)

Mari kita cek bahwa versi satu baris sama dengan versi langkah demi langkah di atas.

In [3]:
layer_2_oneliner = relu(layer_0.dot(weights_0_1)).dot(weights_1_2)

print("langkah demi langkah:", np.round(layer_2, 4))
print("satu baris          :", np.round(layer_2_oneliner, 4))
print("sama?", np.allclose(layer_2, layer_2_oneliner))

langkah demi langkah: [[0.3919]]
satu baris          : [[0.3919]]
sama? True


## 10. Pentingnya Alat Visualisasi

Bab-bab berikutnya akan menggabungkan vektor dan matriks dengan cara yang kreatif. Kemampuan penulis menjelaskan tiap arsitektur bergantung pada bahasa yang disepakati bersama. Karena itu, jangan lanjut sebelum bisa melihat dengan jelas bagaimana forward propagation memanipulasi vektor dan matriks ini, dan bagaimana berbagai bentuk deskripsi tadi saling berhubungan.

> **Kesimpulan utama**: arsitektur neural yang baik mengalirkan sinyal sehingga korelasi mudah ditemukan. Arsitektur yang hebat juga menyaring noise untuk membantu mencegah overfitting.

Arsitektur mengontrol bagaimana sinyal mengalir lewat network, dan cara kita membuatnya memengaruhi kemampuan network mendeteksi korelasi. Kita ingin arsitektur yang memaksimalkan kemampuan network untuk fokus pada area yang punya korelasi bermakna, dan meminimalkan kemampuannya fokus pada area yang berisi noise.

Dataset dan domain yang berbeda punya karakter berbeda. Data gambar memiliki sinyal dan noise yang berbeda dari data teks. Karena itu, arsitektur tertentu lebih cocok untuk masalah tertentu, berdasarkan kemampuannya menemukan jenis korelasi tertentu. Beberapa bab berikutnya akan membahas cara mengubah neural network agar menemukan korelasi yang kita cari.

## Poin-poin Penting Bab 7

- **Correlation summarization**: neural network mencari korelasi langsung maupun tidak langsung antara layer input dan layer output. Gagasan ini merangkum neuron, gradien, dan tumpukan layer jadi satu ide.
- **Local correlation summarization**: tiap set weight belajar mengorelasikan inputnya dengan apa yang menurut layer output seharusnya terjadi.
- **Global correlation summarization**: layer belakang memberi tahu layer depan sinyal apa yang dibutuhkan (backpropagation), jadi kita tidak perlu memikirkan pembaruan weight satu per satu.
- Network bisa digambar sebagai **balok LEGO**: strip untuk vektor, kotak untuk matriks. Dimensi sebuah matriks ditentukan oleh layer di kiri dan kanannya.
- Susunan layer dan weight disebut **arsitektur**. Arsitektur yang baik mengalirkan sinyal agar korelasi mudah ditemukan, dan yang hebat juga menyaring noise.
- Network dapat ditulis sebagai aljabar: matriks = huruf kapital (W), vektor = huruf kecil (l). Seluruh forward propagation menjadi `l2 = relu(l0 W0) W1`, setara dengan `layer_2 = relu(layer_0.dot(weights_0_1)).dot(weights_1_2)`.
- Memahami keempat cara pandang (gambar, aljabar, kode, bentuk matriks) adalah prasyarat sebelum melanjutkan ke arsitektur lain.

Referensi: Trask, A. W. (2019). *Grokking Deep Learning*, Bab 7 "How to Picture Neural Networks: In Your Head and on Paper". Manning Publications.